## Step 1: Extraction

In [1]:
import re
import json
from pathlib import Path

INPUT_PATH = Path("thegreatgatsby.txt")
OUTPUT_PATH = Path("extracted_book.json")

BOOK_ID = "great-gatsby"
BOOK_TITLE = "The Great Gatsby"
AUTHOR = "F. Scott Fitzgerald"

# Known chapter numerals, in order -- used both to detect real chapter
# headings and to filter out the redundant "Table of Contents" listing.
CHAPTER_NUMERALS = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX"]

In [2]:
START_RE = re.compile(r"\*\*\*\s*START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*", re.IGNORECASE)
END_RE = re.compile(r"\*\*\*\s*END OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*", re.IGNORECASE)

# A line containing *only* a chapter numeral (optionally centered with leading/trailing
# whitespace) -- matches both the real chapter heading and the bare Table-of-Contents
# listing, which is handled by keeping the LAST occurrence of each numeral.
CHAPTER_HEADING_RE = re.compile(r"^[ \t]*(" + "|".join(CHAPTER_NUMERALS) + r")[ \t]*$")

# Scene-break divider rendered as a line of dashes in the plain-text edition.
SCENE_BREAK_RE = re.compile(r"^-{5,}$")


def strip_boilerplate(raw_text: str) -> str:
    """Return only the text between the Gutenberg START/END markers."""
    start_match = START_RE.search(raw_text)
    end_match = END_RE.search(raw_text)
    if not start_match or not end_match:
        raise ValueError("Could not locate Project Gutenberg start/end markers")
    return raw_text[start_match.end():end_match.start()]


def split_front_matter_and_chapters(body: str):
    """Split the body into (front_matter_text, [(numeral, chapter_text), ...])."""
    lines = body.split("\n")

    chapter_starts = []
    for i, line in enumerate(lines):
        m = CHAPTER_HEADING_RE.match(line)
        if m:
            chapter_starts.append((m.group(1), i))

    # Each numeral appears twice: once in the bare Table-of-Contents listing,
    # once as the real (centered) chapter heading further down. Keep the later
    # occurrence of each, then order by line position.
    last_seen = {}
    for numeral, idx in chapter_starts:
        last_seen[numeral] = idx

    ordered = [(numeral, last_seen[numeral]) for numeral in CHAPTER_NUMERALS if numeral in last_seen]
    ordered.sort(key=lambda x: x[1])

    front_matter_lines = lines[:ordered[0][1]]
    chapters_raw = []
    for idx, (numeral, start_line) in enumerate(ordered):
        end_line = ordered[idx + 1][1] if idx + 1 < len(ordered) else len(lines)
        chapter_lines = lines[start_line + 1:end_line]
        chapters_raw.append((numeral, "\n".join(chapter_lines)))

    return "\n".join(front_matter_lines), chapters_raw


def split_sections(chapter_body: str):
    """Split a chapter's text into scene-break sections."""
    lines = chapter_body.split("\n")
    sections = [[]]
    for line in lines:
        if SCENE_BREAK_RE.match(line.strip()):
            sections.append([])
        else:
            sections[-1].append(line)
    return ["\n".join(sec) for sec in sections]


def split_paragraphs(section_text: str):
    """Split a section's text into paragraphs on blank lines, collapsing hard-wraps."""
    blocks = re.split(r"\n\s*\n", section_text.strip())
    paragraphs = []
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        collapsed = re.sub(r"\s+", " ", block).strip()
        paragraphs.append(collapsed)
    return paragraphs

In [3]:
# Bare listing of consecutive chapter numerals, e.g. "I II III IV V VI VII VIII IX"
# (the Table-of-Contents block, once its lines are collapsed to one line).
TOC_LIST_RE = re.compile(
    r"^(?:" + "|".join(CHAPTER_NUMERALS) + r")(?:\s+(?:" + "|".join(CHAPTER_NUMERALS) + r"))*$"
)


def extract_front_matter_blocks(front_matter_text: str):
    """Parse the front matter into typed blocks, dropping repeated-title/ToC boilerplate."""
    blocks = re.split(r"\n\s*\n", front_matter_text.strip())
    parsed = []
    for block in blocks:
        raw_lines = [l for l in block.split("\n") if l.strip()]
        if not raw_lines:
            continue
        # Heuristic: every non-blank line indented (centered/verse formatting)
        # vs. body paragraphs, which start flush at column 0.
        is_indented = all(re.match(r"^[ \t]+\S", l) for l in raw_lines)
        text = re.sub(r"\s+", " ", block.strip())
        parsed.append({"type": "verse" if is_indented else "paragraph", "text": text})

    # Drop the repeated running-title line and the Table-of-Contents heading/listing --
    # boilerplate that duplicates book_title/chapters metadata already captured elsewhere.
    filtered = []
    for b in parsed:
        if b["text"] == f"{BOOK_TITLE} by {AUTHOR}":
            continue
        if b["text"] == "Table of Contents":
            continue
        if TOC_LIST_RE.match(b["text"]):
            continue
        filtered.append(b)
    return filtered

In [4]:
def extract_book(path: Path) -> dict:
    raw_text = path.read_text(encoding="utf-8")
    body = strip_boilerplate(raw_text)
    front_matter_text, chapters_raw = split_front_matter_and_chapters(body)

    front_matter_blocks = extract_front_matter_blocks(front_matter_text)

    chapters = []
    for chapter_number, (numeral, chapter_text) in enumerate(chapters_raw, start=1):
        sections_raw = split_sections(chapter_text)
        sections = []
        for section_index, section_text in enumerate(sections_raw):
            paragraphs = split_paragraphs(section_text)
            if not paragraphs:
                continue
            sections.append({
                "section_index": section_index,
                "paragraphs": [
                    {"paragraph_index": p_i, "type": "paragraph", "text": p_text}
                    for p_i, p_text in enumerate(paragraphs)
                ],
            })
        chapters.append({
            "chapter_number": chapter_number,
            "chapter_title": numeral,
            "sections": sections,
        })

    return {
        "book_id": BOOK_ID,
        "book_title": BOOK_TITLE,
        "author": AUTHOR,
        "front_matter": front_matter_blocks,
        "chapters": chapters,
    }


book = extract_book(INPUT_PATH)

### Sanity check

Expect 9 chapters, a small front-matter block list, and a total body word count in the ~47k–50k range (Gatsby's known length).

In [5]:
print(f"Front matter blocks: {len(book['front_matter'])}")
for b in book["front_matter"]:
    print(f"  [{b['type']}] {b['text'][:60]!r}")

print(f"\nChapters: {len(book['chapters'])}")
for ch in book["chapters"]:
    n_paras = sum(len(sec["paragraphs"]) for sec in ch["sections"])
    print(f"  Chapter {ch['chapter_number']:>2} ({ch['chapter_title']:>4}): "
          f"{len(ch['sections'])} sections, {n_paras} paragraphs")

total_words = sum(
    len(p["text"].split())
    for ch in book["chapters"]
    for sec in ch["sections"]
    for p in sec["paragraphs"]
)
print(f"\nTotal body word count: {total_words}")

print("\nSample paragraph (Chapter 1, first):")
print(book["chapters"][0]["sections"][0]["paragraphs"][0]["text"])

print("\nLast paragraph of Chapter 9 (should end the novel):")
print(book["chapters"][-1]["sections"][-1]["paragraphs"][-1]["text"][-120:])

Front matter blocks: 3
  [verse] 'Once again to Zelda'
  [verse] 'Then wear the gold hat, if that will move her; If you can bo'
  [verse] 'Thomas Parke d’Invilliers'

Chapters: 9
  Chapter  1 (   I): 3 sections, 152 paragraphs
  Chapter  2 (  II): 1 sections, 137 paragraphs
  Chapter  3 ( III): 2 sections, 171 paragraphs
  Chapter  4 (  IV): 5 sections, 170 paragraphs
  Chapter  5 (   V): 2 sections, 153 paragraphs
  Chapter  6 (  VI): 2 sections, 135 paragraphs
  Chapter  7 ( VII): 5 sections, 415 paragraphs
  Chapter  8 (VIII): 8 sections, 113 paragraphs
  Chapter  9 (  IX): 6 sections, 164 paragraphs

Total body word count: 48118

Sample paragraph (Chapter 1, first):
In my younger and more vulnerable years my father gave me some advice that I’ve been turning over in my mind ever since.

Last paragraph of Chapter 9 (should end the novel):
So we beat on, boats against the current, borne back ceaselessly into the past.


In [6]:
OUTPUT_PATH.write_text(json.dumps(book, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved extracted book to {OUTPUT_PATH.resolve()}")

Saved extracted book to D:\Code\Learning\ElasticSearch\extracted_book.json


## Step 2: Chunking & Overlap

Input: `extracted_book.json` from Step 1.

What this step does:
1. Load the **bge-m3 tokenizer** (`BAAI/bge-m3`) so chunk sizes are measured in the embedding model's actual tokens, not a word-count proxy.
2. Greedily accumulate consecutive **paragraphs within a section** into a chunk until adding the next paragraph would exceed `TARGET_CHUNK_TOKENS` (~450 tokens). Paragraphs are never split unless a single paragraph alone exceeds `MAX_CHUNK_TOKENS` (~600), in which case it's force-split at sentence boundaries.
3. Chunks never cross a chapter boundary, and by construction never cross a scene-break/section boundary either -- chapter/section identity is preserved per chunk.
4. Each new chunk (after the first in a section) is seeded with a **~15% token overlap**: the trailing paragraph(s) of the previous chunk are carried forward, so a passage split across a chunk boundary still has a matching vector on both sides. `paragraph_ids` lists everything in the chunk; `new_paragraph_ids` excludes the overlap so new content can be told apart from repeated context.
5. Front-matter blocks (dedication/epigraph) become their own tiny chunks, verbatim, never split or merged.
6. Every chunk gets a `prev_chunk_id`/`next_chunk_id` (global reading order across the whole book) so a chunk can be expanded with its neighbors at query time.
7. Result is saved to `chunks.json` for the embedding step.

Note: `MAX_CHUNK_TOKENS` is a soft ceiling rather than an absolute guarantee -- a chunk can occasionally run a little over it because the overlap seed plus the next paragraph can together exceed the cap slightly. That's harmless here since bge-m3 supports sequences up to 8192 tokens.

In [7]:
import json
import re
from collections import Counter
from pathlib import Path
from transformers import AutoTokenizer

BOOK_JSON_PATH = Path("extracted_book.json")
CHUNKS_JSON_PATH = Path("chunks.json")

TOKENIZER_NAME = "BAAI/bge-m3"
TARGET_CHUNK_TOKENS = 450   # soft target: stop accumulating paragraphs once exceeded
MAX_CHUNK_TOKENS = 600      # hard cap: only a single oversized paragraph is force-split above this
OVERLAP_RATIO = 0.15        # ~15% of TARGET_CHUNK_TOKENS carried forward into the next chunk

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)


def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


book = json.loads(BOOK_JSON_PATH.read_text(encoding="utf-8"))

C:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9"\u2018\u2019\u201c\u201d])')


def split_oversized_atom(atom, target_tokens):
    """Force-split a single paragraph that exceeds MAX_CHUNK_TOKENS, at sentence boundaries."""
    sentences = SENTENCE_SPLIT_RE.split(atom["text"]) or [atom["text"]]
    groups = []
    current_sents, current_tokens = [], 0
    for sent in sentences:
        n = count_tokens(sent)
        if current_sents and current_tokens + n > target_tokens:
            groups.append((current_sents, current_tokens))
            current_sents, current_tokens = [], 0
        current_sents.append(sent)
        current_tokens += n
    if current_sents:
        groups.append((current_sents, current_tokens))

    total_parts = len(groups)
    base = {k: v for k, v in atom.items() if k not in ("text", "token_count")}
    return [
        {**base, "text": " ".join(sents), "token_count": tok_count,
         "is_overlap": False, "paragraph_part": f"{i}/{total_parts}"}
        for i, (sents, tok_count) in enumerate(groups, start=1)
    ]


def take_overlap_tail(atom_list, overlap_ratio, target_tokens):
    """Pull trailing paragraphs from the previous chunk to seed the next chunk's overlap."""
    overlap_budget = max(1, round(target_tokens * overlap_ratio))
    tail, running = [], 0
    for atom in reversed(atom_list):
        if running >= overlap_budget and tail:
            break
        tail.insert(0, {**atom, "is_overlap": True})
        running += atom["token_count"]
    return tail


def chunk_section(paragraphs, target_tokens, max_tokens, overlap_ratio):
    """Greedily group a section's paragraphs into overlapping, token-budgeted chunks."""
    atoms = [{**p, "token_count": count_tokens(p["text"])} for p in paragraphs]
    chunks, current, current_tokens = [], [], 0

    def close_chunk():
        nonlocal current, current_tokens
        if current:
            chunks.append(current)
        current, current_tokens = [], 0

    for atom in atoms:
        if atom["token_count"] > max_tokens:
            close_chunk()
            for sub_atom in split_oversized_atom(atom, target_tokens):
                chunks.append([sub_atom])
            continue

        if current and current_tokens + atom["token_count"] > target_tokens:
            close_chunk()
            current = take_overlap_tail(chunks[-1], overlap_ratio, target_tokens)
            current_tokens = sum(a["token_count"] for a in current)

        current.append({**atom, "is_overlap": False})
        current_tokens += atom["token_count"]

    close_chunk()
    return chunks

In [9]:
def build_chunk_records(book, target_tokens=TARGET_CHUNK_TOKENS, max_tokens=MAX_CHUNK_TOKENS, overlap_ratio=OVERLAP_RATIO):
    records = []

    # Front matter: each block is already tiny and self-contained -- one chunk each, no splitting/overlap.
    for i, block in enumerate(book["front_matter"]):
        records.append({
            "chunk_id": f"{book['book_id']}-front{i:02d}",
            "book_id": book["book_id"], "book_title": book["book_title"], "author": book["author"],
            "chapter_number": 0, "chapter_title": "Front Matter", "section_index": 0, "chunk_index": i,
            "type": block["type"], "paragraph_ids": [i], "new_paragraph_ids": [i],
            "text": block["text"], "token_count": count_tokens(block["text"]),
        })

    for ch in book["chapters"]:
        for sec in ch["sections"]:
            atom_chunks = chunk_section(sec["paragraphs"], target_tokens, max_tokens, overlap_ratio)
            for chunk_idx, atoms in enumerate(atom_chunks):
                text = " ".join(a["text"] for a in atoms)
                chunk_type = "verse" if all(a["type"] == "verse" for a in atoms) else "paragraph"
                records.append({
                    "chunk_id": f"{book['book_id']}-ch{ch['chapter_number']:02d}-s{sec['section_index']:02d}-c{chunk_idx:03d}",
                    "book_id": book["book_id"], "book_title": book["book_title"], "author": book["author"],
                    "chapter_number": ch["chapter_number"], "chapter_title": ch["chapter_title"],
                    "section_index": sec["section_index"], "chunk_index": chunk_idx, "type": chunk_type,
                    "paragraph_ids": [a["paragraph_index"] for a in atoms],
                    "new_paragraph_ids": [a["paragraph_index"] for a in atoms if not a["is_overlap"]],
                    "text": text, "token_count": sum(a["token_count"] for a in atoms),
                })

    # Global reading-order stitching, across the whole book.
    for i, rec in enumerate(records):
        rec["global_index"] = i
        rec["prev_chunk_id"] = records[i - 1]["chunk_id"] if i > 0 else None
        rec["next_chunk_id"] = records[i + 1]["chunk_id"] if i < len(records) - 1 else None

    return records


chunks = build_chunk_records(book)
print(f"Built {len(chunks)} chunks")

Built 247 chunks


### Sanity check

Expect chunk token counts clustered around `TARGET_CHUNK_TOKENS`, most chunks (all but the first per section) carrying overlap paragraphs, correct prev/next stitching, and the oversized-paragraph fallback working on a synthetic example (no real Gatsby paragraph is long enough to trigger it).

In [10]:
token_counts = [c["token_count"] for c in chunks]
print(f"token_count -> min {min(token_counts)}, mean {sum(token_counts)/len(token_counts):.1f}, max {max(token_counts)}")
print(f"chunks exceeding MAX_CHUNK_TOKENS ({MAX_CHUNK_TOKENS}): {sum(1 for t in token_counts if t > MAX_CHUNK_TOKENS)}")
print(f"chunks carrying overlap paragraphs: {sum(1 for c in chunks if len(c['new_paragraph_ids']) < len(c['paragraph_ids']))} / {len(chunks)}")

print("\nchunks per chapter:")
per_chapter = Counter(c["chapter_number"] for c in chunks)
for chapter_number in sorted(per_chapter):
    label = "Front Matter" if chapter_number == 0 else f"Chapter {chapter_number}"
    print(f"  {label:<13} {per_chapter[chapter_number]}")

print("\nsample chunk (metadata):")
sample = chunks[10]
print({k: v for k, v in sample.items() if k != "text"})
print("text preview:", sample["text"][:300], "...")

# Confirm overlap paragraphs actually re-appear at the head of the next chunk within the same section
prev, nxt = chunks[10], chunks[11]
shared = set(prev["paragraph_ids"]) & set(nxt["paragraph_ids"])
print(f"\nShared paragraph_ids between consecutive chunks (the overlap): {sorted(shared)}")
assert nxt["prev_chunk_id"] == prev["chunk_id"] and prev["next_chunk_id"] == nxt["chunk_id"]

# Confirm the oversized-paragraph fallback works, even though it never triggers on real Gatsby text
synthetic_text = " ".join(f"This is sentence number {i} in a very long synthetic paragraph." for i in range(120))
synthetic_atom = {"paragraph_index": 999, "type": "paragraph", "text": synthetic_text}
parts = split_oversized_atom(synthetic_atom, TARGET_CHUNK_TOKENS)
print(f"\nSynthetic {count_tokens(synthetic_text)}-token paragraph split into {len(parts)} parts: "
      f"{[p['token_count'] for p in parts]}")

token_count -> min 5, mean 387.2, max 629
chunks exceeding MAX_CHUNK_TOKENS (600): 1
chunks carrying overlap paragraphs: 210 / 247

chunks per chapter:
  Front Matter  3
  Chapter 1     31
  Chapter 2     24
  Chapter 3     30
  Chapter 4     29
  Chapter 5     21
  Chapter 6     21
  Chapter 7     39
  Chapter 8     24
  Chapter 9     25

sample chunk (metadata):
{'chunk_id': 'great-gatsby-ch01-s01-c005', 'book_id': 'great-gatsby', 'book_title': 'The Great Gatsby', 'author': 'F. Scott Fitzgerald', 'chapter_number': 1, 'chapter_title': 'I', 'section_index': 1, 'chunk_index': 5, 'type': 'paragraph', 'paragraph_ids': [8, 9], 'new_paragraph_ids': [9], 'token_count': 461, 'global_index': 10, 'prev_chunk_id': 'great-gatsby-ch01-s01-c004', 'next_chunk_id': 'great-gatsby-ch01-s01-c006'}
text preview: It was a matter of chance that I should have rented a house in one of the strangest communities in North America. It was on that slender riotous island which extends itself due east of New York—a

In [11]:
CHUNKS_JSON_PATH.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {len(chunks)} chunks to {CHUNKS_JSON_PATH.resolve()}")

Saved 247 chunks to D:\Code\Learning\ElasticSearch\chunks.json


## Step 3: Embedding with bge-m3

1. Load `BAAI/bge-m3` via `FlagEmbedding.BGEM3FlagModel` on GPU (`use_fp16=True` -- halves memory/latency with negligible quality loss). The first run downloads the model weights (~2.2GB) from Hugging Face Hub.
2. Batch-encode every chunk's `text` to get:
   - a **dense** vector (1024-dim, cosine-ready -- bge-m3's dense output is already L2-normalized)
   - a **sparse** lexical-weight map (token -> weight) for hybrid BM25-style retrieval
   - (ColBERT/multi-vector output is skipped -- Elasticsearch has no native late-interaction field type, so it's not worth the complexity here)
3. Attach `embedding` and `sparse_embedding` to each chunk record, alongside everything carried over from Step 2.
4. Save the enriched records to `embedded_chunks.json`, ready for Step 4 (indexing into Elasticsearch).

In [7]:
pip install FlagEmbedding

     -------------------------------------- 250.5/250.5 kB 1.2 MB/s eta 0:00:00
     -------------------------------------- 394.3/394.3 kB 4.1 MB/s eta 0:00:00
     -------------------------------------- 559.1/559.1 kB 7.0 MB/s eta 0:00:00
     ------------------------------------- 947.8/947.8 kB 19.9 MB/s eta 0:00:00
     ------------------------------------- 740.6/740.6 kB 45.6 MB/s eta 0:00:00
     --------------------------------------- 12.3/12.3 MB 19.3 MB/s eta 0:00:00
     ------------------------------------- 832.9/832.9 kB 54.8 MB/s eta 0:00:00
     ---------------------------------------- 1.2/1.2 MB 77.2 MB/s eta 0:00:00
     ---------------------------------------- 456.5/456.5 kB ? eta 0:00:00
     ---------------------------------------- 842.9/842.9 kB ? eta 0:00:00
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl (355 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-win_amd64.whl (158 kB)
     -------------


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install ipykernel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
   ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 GB 65.0 MB/s eta 0:00:39
   ---------------------------------------- 0.0/2.5 GB 34.3 MB/s eta 0:01:14
   ---------------------------------------- 0.0/2.5 GB 37.2 MB/s eta 0:01:08
   ---------------------------------------- 0.0/2.5 GB 28.3 MB/s eta 0:01:29
   ---------------------------------------- 0.0/2.5 GB 28.4 MB/s eta 0:01:29
   ---------------------------------------- 0.0/2.5 GB 26.2 MB/s eta 0:01:36
    --------------------------------------- 0.0/2.5 GB 24.7 MB/s eta 0:01:42
    --------------------------------------- 0.0/2.5 GB 23.6 MB/s eta 0:01:46
    --------------------------------------- 0.0/2.5 GB 22.6 MB/s eta 0:01:51
    --------------------------------------- 0.0/2.5 GB 21.6 MB/s eta 0:01:56
    --------------------------------------- 0.0/2.5 GB 21.9 MB/s eta 0:01:54
    ---------------------

In [6]:
import json
from pathlib import Path

import torch
from FlagEmbedding import BGEM3FlagModel

CHUNKS_JSON_PATH = Path("chunks.json")
EMBEDDED_JSON_PATH = Path("embedded_chunks.json")

MODEL_NAME = "BAAI/bge-m3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32          # tune upward while VRAM has headroom -- watch `nvidia-smi` while running
MAX_LENGTH = 8192        # bge-m3's max sequence length; our chunks (~450-600 tokens) are far under this

print(f"torch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Using device: {DEVICE}")

model = BGEM3FlagModel(MODEL_NAME, use_fp16=(DEVICE == "cuda"), device=DEVICE)

d:\Code\Learning\ElasticSearch\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.6.0+cu124, CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Using device: cuda


Loading weights: 100%|██████████| 391/391 [00:02<00:00, 140.58it/s]


In [7]:
chunks = json.loads(CHUNKS_JSON_PATH.read_text(encoding="utf-8"))
texts = [c["text"] for c in chunks]
print(f"Loaded {len(chunks)} chunks to embed")

Loaded 247 chunks to embed


In [8]:
output = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
)

dense_vecs = output["dense_vecs"]              # numpy array, shape (n_chunks, 1024)
sparse_weights = output["lexical_weights"]     # list of {token_id(str): weight(float)} dicts

# Convert sparse keys from raw XLM-RoBERTa token ids to readable subword tokens,
# so the sparse field is inspectable and stable to index in Elasticsearch.
readable_sparse = model.convert_id_to_token(sparse_weights)

for chunk, dense_vec, sparse in zip(chunks, dense_vecs, readable_sparse):
    # fp16 numpy scalars aren't JSON-serializable -- cast to plain Python floats.
    # Also swap "." out of token names: Elasticsearch's rank_features field
    # treats a literal "." as a nested-object path separator (e.g. "3.00" ->
    # object "3" -> field "00"), which breaks indexing for tokens like "."
    # or "3.00" that bge-m3's sparse output naturally produces.
    chunk["embedding"] = [float(x) for x in dense_vec]
    chunk["sparse_embedding"] = {k.replace(".", "·"): float(v) for k, v in sparse.items()}

print("Done embedding all chunks")

Inference Embeddings: 100%|██████████| 8/8 [00:06<00:00,  1.20it/s]

Done embedding all chunks


### Sanity check

Expect a 1024-dim dense vector per chunk with norm ~1.0 (bge-m3 dense output is L2-normalized), and a sparse map whose top-weighted tokens are the chunk's actual salient words.

In [9]:
sample = chunks[10]
print("dense embedding dims:", len(sample["embedding"]))

norm = sum(x ** 2 for x in sample["embedding"]) ** 0.5
print(f"dense embedding L2 norm: {norm:.4f}")

top_sparse = sorted(sample["sparse_embedding"].items(), key=lambda kv: -kv[1])[:10]
print("top sparse tokens:", top_sparse)

print("\nchunk text preview:", sample["text"][:200], "...")

dense embedding dims: 1024
dense embedding L2 norm: 1.0003
top sparse tokens: [('Egg', np.float16(0.2947)), ('egg', np.float16(0.2878)), ('Sound', np.float16(0.261)), ('sby', np.float16(0.2427)), ('communities', np.float16(0.2161)), ('rent', np.float16(0.2048)), ('Island', np.float16(0.2017)), ('island', np.float16(0.1954)), ('gull', np.float16(0.1906)), ('West', np.float16(0.1884))]

chunk text preview: It was a matter of chance that I should have rented a house in one of the strangest communities in North America. It was on that slender riotous island which extends itself due east of New York—and wh ...


In [12]:
EMBEDDED_JSON_PATH.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {len(chunks)} embedded chunks to {EMBEDDED_JSON_PATH.resolve()}")

Saved 247 embedded chunks to D:\Code\Learning\ElasticSearch\embedded_chunks.json
